## Basic RAG with Ollama + ChromaDB

This notebook demonstrates a basic Retrieval-Augmented Generation (RAG) pipeline
using local Ollama models and ChromaDB as the vector store.


In [1]:
import os
import sys
import subprocess
import time

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if IN_COLAB or IN_KAGGLE:
    !pip install git+https://github.com/saikrishna1729/reliablerag.git@rag_pipeline/jithu datasets pandas -q
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    !ollama pull nomic-embed-text-v2-moe:latest
    !ollama pull llama3.1:8b-instruct-q4_K_M

In [2]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import time

import pandas as pd
from datasets import load_dataset
from langchain_core.runnables import RunnableConfig
from langchain_text_splitters import RecursiveCharacterTextSplitter

from reliablerag.chain import build_rag_chain, TimingCallbackHandler
from reliablerag.env import load_secrets
from reliablerag.evaluation import evaluate
from reliablerag.experiment import evaluate_results, run_rag_experiment
from reliablerag.providers import create_embeddings, create_llm
from reliablerag.retriever import get_hyde_retriever, get_hybrid_retriever, get_hybrid_reranked_retriever, get_or_build_vector_store, get_reranked_retriever, get_reranker, get_retriever

### 1. Configuration

Model names and paths are loaded from `.env`. Fallback defaults are used if not set.

In [3]:
load_secrets()

PROVIDER           = os.environ["PROVIDER"]
EMBEDDING_MODEL    = os.environ["EMBEDDING_MODEL"]
GENERATOR_MODEL    = os.environ["GENERATOR_MODEL"]
JUDGE_MODEL        = os.environ["JUDGE_MODEL"]
CHROMA_PERSIST_DIR = os.environ["CHROMA_PERSIST_DIR"]

print(f"Provider        : {PROVIDER}")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Generator model : {GENERATOR_MODEL}")
print(f"Judge model     : {JUDGE_MODEL}")
print(f"Chroma dir      : {CHROMA_PERSIST_DIR}")

Provider        : ollama
Embedding model : nomic-embed-text-v2-moe:latest
Generator model : gemma4:12b-it-q4_K_M
Judge model     : llama3.1:8b-instruct-q4_K_M
Chroma dir      : /Users/jithamanyu.manne/git/others/python/reliablerag/data/chroma_db


In [4]:
embeddings = create_embeddings(PROVIDER, EMBEDDING_MODEL)

# Generator: large model used to write the final answer.
llm = create_llm(PROVIDER, GENERATOR_MODEL)

# Judge: smaller model used by the TRACe eval. Deterministic decoding so
# repeat evals on the same inputs don't drift. Bumped to a smaller open-source
# model (Llama 3.1 8B) — gemma 12B as judge was making each eval call ~3-4min.
judge_llm = create_llm(PROVIDER, JUDGE_MODEL, temperature=0)

# Cross-encoder reranker. Built once outside the per-sample loop — the
# model weights (~280MB for bge-reranker-base) are downloaded on first use.
reranker = get_reranker()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### 2. Load CUAD Samples from RAGBench

Each sample contains a legal contract (`documents`), a `question`, a reference `response` generated
by Claude 3 Haiku, and pre-computed **TRACe labels** annotated by GPT-4:
`adherence`, `context_relevance`, `utilization`, `completeness`.

In [5]:
N_SAMPLES = 20  # increase to evaluate more samples

dataset = load_dataset("galileo-ai/ragbench", "cuad", split="train")
samples = list(dataset.select(range(N_SAMPLES)))


def fmt(v):
    return f"{v:.3f}" if v is not None else "N/A"


print(f"Loaded {len(samples)} CUAD samples")

s = samples[0]
print(f"\nQuestion         : {s['question']}")
print(f"Doc length       : {len(s['documents'][0])} chars")
print(f"Adherence score  : {s['adherence_score']}")
print(f"Relevance score  : {fmt(s['relevance_score'])}")
print(f"Utilization score: {fmt(s['utilization_score'])}")
print(f"Completeness     : {fmt(s['completeness_score'])}")
print(f"\nAlso available   : ragas_faithfulness={fmt(s['ragas_faithfulness'])}, "
      f"trulens_groundedness={fmt(s['trulens_groundedness'])}")

Loaded 20 CUAD samples

Question         : Is one party required to deposit its source code into escrow with a third party, which can be released to the counterparty upon the occurrence of certain events (bankruptcy,  insolvency, etc.)?
Doc length       : 122054 chars
Adherence score  : True
Relevance score  : 0.000
Utilization score: 0.000
Completeness     : 1.000

Also available   : ragas_faithfulness=N/A, trulens_groundedness=N/A


In [9]:
df_preview = dataset.to_pandas()
# df_preview.head(10)

### 3. Run RAG on Each Sample

CUAD contracts are up to 11k tokens each, so we chunk each document before indexing.
We build a fresh ephemeral vector store per sample (CUAD has 1 doc per question, so no cross-contamination).

In [ ]:
# Exp E — Dense-only baseline (cosine, nomic, chunk_size=500, top_k=20, N=20).
# This is the authoritative baseline all subsequent experiments are measured against.
# Vector store tag cs500_co50 is shared with Steps H–K (cache reused, no re-embedding).
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

results_e = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_retriever(vs, top_k=TOP_K),
    embeddings=embeddings,
    llm=llm,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
)

In [ ]:
# Exp E — TRACe evaluation (dense-only baseline)
JUDGE_N_RUNS = 3
agg_e = evaluate_results(results_e, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Exp E (dense baseline) — Rel {agg_e['avg_relevance']:.3f} / Util {agg_e['avg_utilization']:.3f} / Comp {agg_e['avg_completeness']:.3f} / Adh {agg_e['adherence_rate']:.0%}")
print(f"Ref   (GPT-4 labels)   — Rel 0.069 / Util 0.042 / Comp 0.717 / Adh 90%")

In [ ]:
# Step H — Hybrid retrieval (BM25 + dense, RRF fusion).
# Hypothesis: BM25 recovers exact-term clause misses that dense retrieval drops.
# Baseline (Step F): (0.173 / 0.097 / 0.564 / 55%) — completeness is the gap vs ref 0.717.
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

results = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hybrid_retriever(vs, chunks, top_k=TOP_K),
    embeddings=embeddings,
    llm=llm,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
)

In [ ]:
# Step H — TRACe evaluation
JUDGE_N_RUNS = 3
agg = evaluate_results(results, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Step H (hybrid, bm25=0.5)   — Rel {agg['avg_relevance']:.3f} / Util {agg['avg_utilization']:.3f} / Comp {agg['avg_completeness']:.3f} / Adh {agg['adherence_rate']:.0%}")
print(f"Baseline (dense only)       — Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%")

In [ ]:
# Step I — Tune RRF weights (bm25_weight=0.3, dense_weight=0.7).
# Step H (equal-weight hybrid) beat baseline completeness (0.564 → 0.590) but dropped adherence (55% → 30%).
# Hypothesis: lower BM25 weight keeps coverage while reducing noise.
# Step H (bm25=0.5): Rel 0.112 / Util 0.086 / Comp 0.590 / Adh 30%
# Baseline:          Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
BM25_WEIGHT    = 0.3
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

results_i = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hybrid_retriever(vs, chunks, top_k=TOP_K, bm25_weight=BM25_WEIGHT),
    embeddings=embeddings,
    llm=llm,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
)

In [ ]:
# Step I — TRACe evaluation (bm25_weight=0.3)
JUDGE_N_RUNS = 3
agg_i = evaluate_results(results_i, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Step I  (bm25={BM25_WEIGHT})          — Rel {agg_i['avg_relevance']:.3f} / Util {agg_i['avg_utilization']:.3f} / Comp {agg_i['avg_completeness']:.3f} / Adh {agg_i['adherence_rate']:.0%}")
print(f"Step H  (bm25=0.5)          — Rel 0.112 / Util 0.086 / Comp 0.590 / Adh 30%")
print(f"Baseline (dense only)       — Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%")

In [ ]:
# Step J — Reranker on top of equal-weight hybrid.
# Step I showed weight tuning can't simultaneously improve completeness and adherence.
# Hypothesis: cross-encoder reranker filters BM25 noise *after* retrieval.
# Step I  (bm25=0.3): Rel 0.132 / Util 0.054 / Comp 0.524 / Adh 40%
# Step H  (bm25=0.5): Rel 0.112 / Util 0.086 / Comp 0.590 / Adh 30%
# Baseline:           Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%
CHUNK_SIZE, CHUNK_OVERLAP, FETCH_K, TOP_N = 500, 50, 40, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

reranker = get_reranker()

results_j = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hybrid_reranked_retriever(vs, chunks, reranker, fetch_k=FETCH_K, top_n=TOP_N),
    embeddings=embeddings,
    llm=llm,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
    retrieve_label=f"retrieve+rerank (fetch={FETCH_K}→top-{TOP_N})",
)

In [ ]:
# Step J — TRACe evaluation (hybrid + cross-encoder reranker)
JUDGE_N_RUNS = 3
agg_j = evaluate_results(results_j, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Step J (hybrid+reranker)    — Rel {agg_j['avg_relevance']:.3f} / Util {agg_j['avg_utilization']:.3f} / Comp {agg_j['avg_completeness']:.3f} / Adh {agg_j['adherence_rate']:.0%}")
print(f"Step I (hybrid, bm25=0.3)   — Rel 0.132 / Util 0.054 / Comp 0.524 / Adh 40%")
print(f"Step H (hybrid, bm25=0.5)   — Rel 0.112 / Util 0.086 / Comp 0.590 / Adh 30%")
print(f"Baseline (dense only)       — Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%")

In [ ]:
# Step K — HyDE (Hypothetical Document Embeddings).
# Vocabulary mismatch root cause: embed a hypothetical clause instead of the raw query.
# Step J (hybrid+reranker): Rel 0.135 / Util 0.091 / Comp 0.479 / Adh 20%
# Baseline (dense only):    Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

results_k = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings, llm, top_k=TOP_K),
    embeddings=embeddings,
    llm=llm,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
    retrieve_label=f"hyde retrieve (top-{TOP_K})",
)

In [ ]:
# Step K — TRACe evaluation (HyDE)
JUDGE_N_RUNS = 3
agg_k = evaluate_results(results_k, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Step K  (HyDE)              — Rel {agg_k['avg_relevance']:.3f} / Util {agg_k['avg_utilization']:.3f} / Comp {agg_k['avg_completeness']:.3f} / Adh {agg_k['adherence_rate']:.0%}")
print(f"Step J  (hybrid+reranker)   — Rel 0.135 / Util 0.091 / Comp 0.479 / Adh 20%")
print(f"Baseline (dense only)       — Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%")

In [ ]:
# Step M1 — Purpose-built sentence encoder (BAAI/bge-large-en-v1.5) + HyDE.
# Hypothesis: bge-large is contrastive-trained for cosine retrieval — better geometry than nomic.
# Step K (HyDE + nomic): Rel 0.334 / Util 0.194 / Comp 0.578 / Adh 25%
# Baseline:              Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter          = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
embeddings_bge    = create_embeddings("huggingface", "BAAI/bge-large-en-v1.5")
COLLECTION_TAG_M1 = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}_bge"

results_m1 = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings_bge, llm, top_k=TOP_K),
    embeddings=embeddings_bge,
    llm=llm,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG_M1,
    retrieve_label=f"hyde retrieve (top-{TOP_K})",
)

In [ ]:
# Step M1 — TRACe evaluation (bge-large-en-v1.5 + HyDE)
JUDGE_N_RUNS = 3
agg_m1 = evaluate_results(results_m1, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Step M1 (bge-large + HyDE)  — Rel {agg_m1['avg_relevance']:.3f} / Util {agg_m1['avg_utilization']:.3f} / Comp {agg_m1['avg_completeness']:.3f} / Adh {agg_m1['adherence_rate']:.0%}")
print(f"Step K  (HyDE + nomic)      — Rel 0.334 / Util 0.194 / Comp 0.578 / Adh 25%")
print(f"Baseline (dense only)       — Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%")

In [ ]:
# Step M2 — Legal-domain BERT (nlpaueb/legal-bert-base-uncased) + HyDE.
# Hypothesis: legal-bert vocabulary matches CUAD contracts better despite CLS-pooling.
# Step M1 (bge-large + HyDE): Rel 0.191 / Util 0.083 / Comp 0.557 / Adh 45%
# Step K  (HyDE + nomic):     Rel 0.334 / Util 0.194 / Comp 0.578 / Adh 25%
# Baseline:                   Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter          = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
embeddings_legal  = create_embeddings("huggingface", "nlpaueb/legal-bert-base-uncased")
COLLECTION_TAG_M2 = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}_legalbert"

results_m2 = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings_legal, llm, top_k=TOP_K),
    embeddings=embeddings_legal,
    llm=llm,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG_M2,
    retrieve_label=f"hyde retrieve (top-{TOP_K})",
)

In [ ]:
# Step M2 — TRACe evaluation (legal-bert-base-uncased + HyDE)
JUDGE_N_RUNS = 3
agg_m2 = evaluate_results(results_m2, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Step M2 (legal-bert + HyDE) — Rel {agg_m2['avg_relevance']:.3f} / Util {agg_m2['avg_utilization']:.3f} / Comp {agg_m2['avg_completeness']:.3f} / Adh {agg_m2['adherence_rate']:.0%}")
print(f"Step M1 (bge-large + HyDE)  — Rel 0.191 / Util 0.083 / Comp 0.557 / Adh 45%")
print(f"Step K  (HyDE + nomic)      — Rel 0.334 / Util 0.194 / Comp 0.578 / Adh 25%")
print(f"Baseline (dense only)       — Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%")